In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from google.colab import files
import pandas as pd
import numpy as np
import torch

In [2]:
uploaded = files.upload()

Saving tweets_2020_cleaned.csv to tweets_2020_cleaned.csv
Saving tweets_2024_cleaned.csv to tweets_2024_cleaned.csv


In [3]:
tweet_2020_df = pd.read_csv("tweets_2020_cleaned.csv")

print (tweet_2020_df.shape)
tweet_2020_df.head()

(12906, 4)


,tweet_id,text,label,year
0,1.324681e+18,hey : you are fired. we the people chose presi...,democrat,2020
1,1.316964e+18,it's sad when even biden gets 🤥🤥🤥🤥🤥 from polit...,democrat,2020
2,1.322311e+18,biden harris trump2020 constitution,democrat,2020
3,1.323692e+18,biden racist pedophile,democrat,2020
4,1.318718e+18,"for better or for worse, the 2 biggest issues ...",democrat,2020


In [4]:
tweet_2024_df = pd.read_csv("tweets_2024_cleaned.csv")

print (tweet_2024_df.shape)
tweet_2020_df.head()

(15708, 4)


,tweet_id,text,label,year
0,1.324681e+18,hey : you are fired. we the people chose presi...,democrat,2020
1,1.316964e+18,it's sad when even biden gets 🤥🤥🤥🤥🤥 from polit...,democrat,2020
2,1.322311e+18,biden harris trump2020 constitution,democrat,2020
3,1.323692e+18,biden racist pedophile,democrat,2020
4,1.318718e+18,"for better or for worse, the 2 biggest issues ...",democrat,2020


### TF-IDF embedding + logistic regression approach

In [19]:
tweets_2020 = tweet_2020_df["text"]
labels_2020 = tweet_2020_df["label"]
tweets_2024 = tweet_2024_df["text"]
labels_2024 = tweet_2024_df["label"]

X_train_tfidf_20, X_test_tfidf_20, y_train_tfidf_20, y_test_tfidf_20 = train_test_split(tweets_2020, labels_2020, test_size=0.2)
X_train_tfidf_24, X_test_tfidf_24, y_train_tfidf_24, y_test_tfidf_24 = train_test_split(tweets_2024, labels_2024, test_size=0.2)

print ("2020 tweet dataset:")
print (X_train_tfidf_20.shape, X_test_tfidf_20.shape, y_train_tfidf_20.shape, y_test_tfidf_20.shape)
print ("\n2024 tweet dataset:")
print (X_train_tfidf_24.shape, X_test_tfidf_24.shape, y_train_tfidf_24.shape, y_test_tfidf_24.shape)

2020 tweet dataset:
(10324,) (2582,) (10324,) (2582,)

2024 tweet dataset:
(12566,) (3142,) (12566,) (3142,)


In [28]:
tfidf_vectorizer_20 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words="english")
tfidf_train_20 = tfidf_vectorizer_20.fit_transform(X_train_tfidf_20)
tfidf_test_20 = tfidf_vectorizer_20.transform(X_test_tfidf_20)

tfidf_vectorizer_24 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words="english")
tfidf_train_24 = tfidf_vectorizer_24.fit_transform(X_train_tfidf_24)
tfidf_test_24 = tfidf_vectorizer_24.transform(X_test_tfidf_24)

print ("2020 tweet training and testing embedding shape:")
print (tfidf_train_20.shape, tfidf_test_20.shape)
print ("\n2024 tweet training and testing embedding shape:")
print (tfidf_train_24.shape, tfidf_test_24.shape)

2020 tweet training and testing embedding shape:
(10324, 10000) (2582, 10000)

2024 tweet training and testing embedding shape:
(12566, 10000) (3142, 10000)


In [29]:
log_regress_model_20 = LogisticRegression()
log_regress_model_20.fit(tfidf_train_20, y_train_tfidf_20)
log_regress_pred_20 = log_regress_model_20.predict(tfidf_test_20)

log_regress_model_24 = LogisticRegression()
log_regress_model_24.fit(tfidf_train_24, y_train_tfidf_24)
log_regress_pred_24 = log_regress_model_24.predict(tfidf_test_24)

print ("TF-IDF + logistic regression result")
print ("2020 tweet prediction:")
print (f"{len(log_regress_pred_20)} predictions")
print (f"First 10 predictions: {log_regress_pred_20[:10]}")
print ("\n2024 tweet prediction:")
print (f"{len(log_regress_pred_24)} predictions")
print (f"First 10 predictions: {log_regress_pred_24[:10]}")

TF-IDF + logistic regression result
2020 tweet prediction:
2582 predictions
First 10 predictions: ['democrat' 'democrat' 'republican' 'republican' 'republican' 'democrat'
 'democrat' 'democrat' 'democrat' 'republican']

2024 tweet prediction:
3142 predictions
First 10 predictions: ['democrat' 'democrat' 'democrat' 'democrat' 'republican' 'republican'
 'democrat' 'democrat' 'democrat' 'democrat']


In [33]:
log_regress_report_20 = classification_report(y_test_tfidf_20, log_regress_pred_20)
log_regress_report_24 = classification_report(y_test_tfidf_24, log_regress_pred_24)

print ("TF-IDF + logistic regression result")
print ("2020 tweet prediction report:")
print (log_regress_report_20)
print ("\n2024 tweet prediction report:")
print (log_regress_report_24)

TF-IDF + logistic regression result
2020 tweet prediction report:
              precision    recall  f1-score   support

    democrat       0.85      0.88      0.87      1233
  republican       0.89      0.86      0.87      1349

    accuracy                           0.87      2582
   macro avg       0.87      0.87      0.87      2582
weighted avg       0.87      0.87      0.87      2582


2024 tweet prediction report:
              precision    recall  f1-score   support

    democrat       0.90      0.93      0.91      1578
  republican       0.92      0.90      0.91      1564

    accuracy                           0.91      3142
   macro avg       0.91      0.91      0.91      3142
weighted avg       0.91      0.91      0.91      3142



### DistilBERT approach

In [37]:
num_classes = len(tweet_2020_df["label"].unique())
bert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_classes)

if torch.cuda.is_available():
  print ("Using GPU")
  bert_model.to("cuda")
else:
  print ("Using CPU")
  bert_model.to("cpu")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using GPU
